In [1]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

In [2]:
# Loading dataset
import tensorflow as tf
from tensorflow import keras

(X_train, Y_train), (X_test, Y_test) = keras.datasets.mnist.load_data()

In [3]:
# Scaling the values from 0-255 to 0-1
X_train = X_train/255
X_test = X_test/255

In [4]:
# Model creation and training
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(100, activation='relu'),
    layers.Dense(10, activation='sigmoid') # Changed to softmax for multi-class
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy']
             )

model.fit(X_train, Y_train, epochs=5)

Epoch 1/5
1875/1875 [==============================] - 9s 4ms/step - loss: 0.2734 - accuracy: 0.9226
Epoch 2/5
1875/1875 [==============================] - 4s 2ms/step - loss: 0.1256 - accuracy: 0.9634
Epoch 3/5
1875/1875 [==============================] - 5s 3ms/step - loss: 0.0884 - accuracy: 0.9736
Epoch 4/5
1875/1875 [==============================] - 4s 2ms/step - loss: 0.0664 - accuracy: 0.9798
Epoch 5/5
1875/1875 [==============================] - 4s 2ms/step - loss: 0.0519 - accuracy: 0.9843


In [5]:
model.evaluate(X_test, Y_test)

313/313 [==============================] - 1s 3ms/step - loss: 0.0770 - accuracy: 0.9767


[0.0769803375005722, 0.9767000079154968]

In [6]:
# Saving model
import os
if not os.path.exists('./saved_model'):
    os.makedirs('./saved_model')
model.save("./saved_model/model.keras")

# Post training quantization

### **Without quantization**

In [7]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

In [8]:
len(tflite_model)

319920

In [9]:
with open('./saved_model/tflite_model.tflite', 'wb') as f:
    f.write(tflite_model)

### **With quantization**

In [10]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()

In [11]:
len(tflite_quant_model)

85984

In [12]:
with open('./saved_model/tflite_quant_model.tflite', 'wb') as f:
    f.write(tflite_quant_model)

# Quantization aware training

In [13]:
!pip install -q tensorflow_model_optimization

In [14]:
import tensorflow_model_optimization as tfmot

quantize_model = tfmot.quantization.keras.quantize_model
q_aware_model = quantize_model(model)

q_aware_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

q_aware_model.fit(X_train, Y_train, epochs=1)

1875/1875 [==============================] - 17s 8ms/step - loss: 0.0439 - accuracy: 0.9869


In [16]:
converter = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_qaware_model = converter.convert()

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [19]:
len(tflite_qaware_model)

82696

In [20]:
with open("./saved_model/tflite_qaware_model.tflite", "wb") as f:
    f.write(tflite_qaware_model)